# IMDB Movie Review Sentiment Dataset — GCS Data Preparation

**Project:** Movie Review Sentiment Classifier  
**Dataset:** IMDB 50,000 movie reviews  
**Task:** Binary sentiment classification — `positive` vs `negative`

This notebook prepares the IMDB dataset for downstream machine-learning work and stores the resulting datasets in Google Cloud Storage (GCS).

### Workflow

1. Authenticate with Google Cloud
2. Configure the GCS project and bucket
3. Download the raw IMDB dataset from GCS
4. Validate the dataset
5. Create stratified **80% train / 10% dev / 10% test** splits
6. Create **8-fold stratified cross-validation** folds from the training set only
7. Upload the partitions to GCS
8. Verify the final GCS structure and a sample shard


## 1. Import Libraries and Configure GCP

The notebook uses Google Cloud Storage for dataset persistence and scikit-learn for reproducible stratified splitting.


In [ ]:
import os
import pandas as pd

from google.colab import auth
from google.cloud import storage
from sklearn.model_selection import train_test_split, StratifiedKFold

# Authenticate the current Colab session with Google Cloud.
auth.authenticate_user()

# -----------------------------
# GCP configuration
# -----------------------------
PROJECT_ID = "dh-and-infra-for-ai"
BUCKET_NAME = "imdb-sentiment-project"
LOCATION = "EU"

# GCS locations
RAW_GCS_PATH = "raw/IMDB Dataset.csv"
SHARDED_PREFIX = "sharded"

# Local working paths
LOCAL_RAW_PATH = "IMDB Dataset.csv"
LOCAL_SHARDS_FOLDER = "shards"

print(f"Project: {PROJECT_ID}")
print(f"Bucket: gs://{BUCKET_NAME}")
print(f"Raw object: gs://{BUCKET_NAME}/{RAW_GCS_PATH}")


## 2. Connect to the Existing GCS Bucket

This project already uses the bucket `imdb-sentiment-project`. The notebook checks that the bucket is accessible before continuing.


In [ ]:
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)

if not storage_client.lookup_bucket(BUCKET_NAME):
    raise ValueError(
        f"Bucket gs://{BUCKET_NAME} was not found or is not accessible."
    )

print(f"Connected successfully to gs://{BUCKET_NAME}")


## 3. Download the Raw IMDB Dataset from GCS

The raw dataset is kept under `raw/` and is **not overwritten** after the train/dev/test split.

This is important because the raw object should remain the original source dataset, while derived datasets are stored separately under `sharded/`.


In [ ]:
blob = bucket.blob(RAW_GCS_PATH)

if not blob.exists():
    raise FileNotFoundError(
        f"Raw dataset not found at gs://{BUCKET_NAME}/{RAW_GCS_PATH}"
    )

blob.download_to_filename(LOCAL_RAW_PATH)

print(f"Downloaded gs://{BUCKET_NAME}/{RAW_GCS_PATH}")
print(f"Saved locally as: {LOCAL_RAW_PATH}")
print(f"File size: {os.path.getsize(LOCAL_RAW_PATH):,} bytes")


## 4. Load and Validate the IMDB Dataset

The expected schema is:

- `review` — the movie-review text
- `sentiment` — the target label (`positive` or `negative`)

Before splitting, we check the shape, column names, missing values, duplicates, and class distribution.


In [ ]:
df = pd.read_csv(
    LOCAL_RAW_PATH,
    encoding="utf-8"
)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

expected_columns = {"review", "sentiment"}
missing_columns = expected_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

# Keep only the expected project columns if extra columns are present.
df = df[["review", "sentiment"]].copy()

# Basic data-quality checks
print("\nMissing values:")
print(df.isna().sum())

print(f"\nDuplicate reviews: {df['review'].duplicated().sum()}")

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())
print("\nSentiment proportions:")
print(df["sentiment"].value_counts(normalize=True).round(4))

display(df.head())


## 5. Create Stratified Train / Dev / Test Splits

The dataset is split into:

- **80% training**
- **10% development/validation**
- **10% test**

Stratification preserves the positive/negative class proportions across the three partitions.

The random seed is fixed at `42` so the split can be reproduced.


In [ ]:
RANDOM_STATE = 42

# First split: 80% train, 20% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df["sentiment"]
)

# Second split: divide the 20% temporary set equally into
# 10% development and 10% test.
dev_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["sentiment"]
)

print(f"Train: {len(train_df):,} rows ({len(train_df)/len(df):.1%})")
print(f"Dev:   {len(dev_df):,} rows ({len(dev_df)/len(df):.1%})")
print(f"Test:  {len(test_df):,} rows ({len(test_df)/len(df):.1%})")


## 6. Validate the Split Distributions

This check confirms that each partition contains both sentiment classes with approximately the same class proportions.


In [ ]:
def show_split_distribution(name, data):
    counts = data["sentiment"].value_counts()
    proportions = data["sentiment"].value_counts(normalize=True)

    print(f"--- {name} ---")
    print(f"Rows: {len(data):,}")
    print("Counts:")
    print(counts.to_dict())
    print("Proportions:")
    print(proportions.round(4).to_dict())
    print()

show_split_distribution("Train", train_df)
show_split_distribution("Dev", dev_df)
show_split_distribution("Test", test_df)


## 7. Create 8-Fold Stratified Cross-Validation

Cross-validation is generated from the **training set only**.

The development and test sets are not included in these folds. This keeps the final evaluation data isolated from model-selection activity and reduces evaluation leakage.


In [ ]:
skf = StratifiedKFold(
    n_splits=8,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_folds = {}

for fold_idx, (train_idx, val_idx) in enumerate(
    skf.split(train_df, train_df["sentiment"])
):
    fold_train = train_df.iloc[train_idx].copy()
    fold_val = train_df.iloc[val_idx].copy()

    cv_folds[fold_idx] = (fold_train, fold_val)

print(f"Number of CV folds: {len(cv_folds)}")

for fold_idx, (fold_train, fold_val) in cv_folds.items():
    print(
        f"Fold {fold_idx}: "
        f"train={len(fold_train):,}, "
        f"validation={len(fold_val):,}"
    )


## 8. Prepare Local GCS Shards

The final GCS structure will be:

```text
gs://imdb-sentiment-project/
├── raw/
│   └── IMDB Dataset.csv
└── sharded/
    ├── train.csv
    ├── dev.csv
    ├── test.csv
    └── cv_folds/
        ├── fold_0_train.csv
        ├── fold_0_val.csv
        ├── ...
        ├── fold_7_train.csv
        └── fold_7_val.csv
```

The raw dataset remains unchanged.


In [ ]:
# Create local output directories.
os.makedirs(f"{LOCAL_SHARDS_FOLDER}/cv_folds", exist_ok=True)

# Save train/dev/test partitions.
train_df.to_csv(
    f"{LOCAL_SHARDS_FOLDER}/train.csv",
    index=False
)

dev_df.to_csv(
    f"{LOCAL_SHARDS_FOLDER}/dev.csv",
    index=False
)

test_df.to_csv(
    f"{LOCAL_SHARDS_FOLDER}/test.csv",
    index=False
)

# Save the eight CV folds.
for fold_idx, (fold_train, fold_val) in cv_folds.items():
    fold_train.to_csv(
        f"{LOCAL_SHARDS_FOLDER}/cv_folds/fold_{fold_idx}_train.csv",
        index=False
    )

    fold_val.to_csv(
        f"{LOCAL_SHARDS_FOLDER}/cv_folds/fold_{fold_idx}_val.csv",
        index=False
    )

print("Local train/dev/test and CV shards created.")


## 9. Upload Shards to GCS

Only the derived partitions are uploaded under the `sharded/` prefix.

A reusable upload function keeps the code concise and avoids repeating the upload logic for every file.


In [ ]:
def upload_folder_to_gcs(local_folder, bucket_name, gcs_prefix):
    """Recursively upload all files under local_folder to GCS."""
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)

    uploaded = 0

    for root, _, files in os.walk(local_folder):
        for file_name in sorted(files):
            local_path = os.path.join(root, file_name)

            relative_path = os.path.relpath(
                local_path,
                local_folder
            ).replace(os.sep, "/")

            gcs_blob_path = f"{gcs_prefix}/{relative_path}"

            blob = bucket.blob(gcs_blob_path)
            blob.upload_from_filename(local_path)

            print(f"Uploaded: gs://{bucket_name}/{gcs_blob_path}")
            uploaded += 1

    return uploaded


uploaded_count = upload_folder_to_gcs(
    LOCAL_SHARDS_FOLDER,
    BUCKET_NAME,
    SHARDED_PREFIX
)

print(f"\nUploaded {uploaded_count} shard files successfully.")


## 10. Verify the GCS Bucket Structure

This final check lists the objects stored in the bucket and confirms that a CV validation shard can be read directly from GCS.


In [ ]:
# List objects in the bucket.
storage_client = storage.Client(project=PROJECT_ID)
blobs = storage_client.list_blobs(BUCKET_NAME)

print(f"--- Objects in gs://{BUCKET_NAME} ---")

object_names = []
for blob in blobs:
    object_names.append(blob.name)
    print(f"{blob.name} ({blob.size:,} bytes)")

# Check the expected key objects.
required_objects = [
    RAW_GCS_PATH,
    f"{SHARDED_PREFIX}/train.csv",
    f"{SHARDED_PREFIX}/dev.csv",
    f"{SHARDED_PREFIX}/test.csv",
    f"{SHARDED_PREFIX}/cv_folds/fold_0_train.csv",
    f"{SHARDED_PREFIX}/cv_folds/fold_0_val.csv",
]

missing_objects = [
    obj for obj in required_objects
    if obj not in object_names
]

if missing_objects:
    raise ValueError(
        f"Missing expected GCS objects: {missing_objects}"
    )

print("\nAll required GCS objects are present.")


## 11. Read a CV Shard Directly from GCS

This smoke test verifies that the generated data can be consumed programmatically from GCS.

If `gcsfs` is not already available in your Colab environment, install it once with:

```python
!pip install -q gcsfs
```


In [ ]:
GCS_CV_PATH = (
    f"gs://{BUCKET_NAME}/"
    f"{SHARDED_PREFIX}/cv_folds/fold_0_val.csv"
)

fold_0_val_df = pd.read_csv(GCS_CV_PATH)

print(f"Read from: {GCS_CV_PATH}")
print(f"Rows loaded: {len(fold_0_val_df):,}")
print(f"Columns: {fold_0_val_df.columns.tolist()}")

display(fold_0_val_df.head(3))


## 12. Final Summary

The notebook has now completed the GCS data-preparation workflow:

- Raw IMDB data is stored under `raw/`.
- The dataset is split into **80% train / 10% dev / 10% test** using stratification.
- **8-fold stratified cross-validation** is generated from the training set only.
- Train/dev/test and CV shards are stored under `sharded/`.
- The final GCS structure is verified.
- A CV shard is read directly from GCS.

### Important design decision

The raw dataset is intentionally **not overwritten with a `split` column**. Keeping raw and derived data separate makes the pipeline easier to reproduce and prevents accidental modification of the source dataset.


# Recommended Improvements for the Next Version

These are improvements to consider after the cleaned storage notebook is working.

### 1. Add dataset versioning
Instead of always writing to the same paths, consider:

```text
raw/v1/IMDB Dataset.csv
sharded/v1/train.csv
sharded/v1/dev.csv
sharded/v1/test.csv
```

This prevents accidental overwriting and makes experiments reproducible.

### 2. Add a stable record ID
Create a unique `review_id` before splitting. This makes it easier to trace a review across preprocessing, training, evaluation, and error analysis.

### 3. Add stronger data-quality checks
For example:

- null reviews
- empty reviews
- duplicate reviews
- unexpected sentiment labels
- duplicate `review_id` values
- train/dev/test overlap

### 4. Consider Parquet for larger downstream workflows
CSV is simple and suitable for this project, but Parquet is generally more efficient for analytical workloads.

### 5. Keep model development separate
Use this notebook for **data ingestion and GCS preparation**. A second notebook can handle:

- text cleaning
- TF-IDF/vectorisation
- Logistic Regression / SVM
- cross-validation
- dev-set model selection
- final test evaluation
- confusion matrix
- precision, recall and F1-score
- saving the final model/vectorizer

### 6. Avoid storing credentials in the notebook
Continue using Colab authentication/IAM rather than embedding service-account keys or passwords.

### 7. Add a reproducibility section
Record:

- random seed
- Python/package versions
- dataset version
- split strategy
- CV strategy
- model configuration

This will make the project easier to explain in an academic report or presentation.
